
# Beer PCA 실험 노트북 (화학 feature 상관 제거 → 회귀 모델)

이 노트북은 **선형대수 + 확률통계 관점**으로 PCA를 사용해:

1. (train 데이터 기준) 화학 feature들의 **공분산/상관 구조**를 분석하고  
2. **PCA 축(고유벡터)**으로 좌표계를 바꿔서 **새 feature(PC score)들끼리 공분산이 0(=비상관)**이 되도록 만든 뒤  
3. 그 PC score로 관능평가(타깃)를 예측하는 회귀 모델(PCR 등)을 학습/평가하는 실험 흐름을 제공합니다.
4. 마지막으로 **train에서 학습된 PCA 축과 스케일러를 저장(joblib)**해서, 이후 같은 축으로 언제든지 변환(transform)할 수 있게 합니다.

---

## PCA를 “선형대수 + 확률통계”로만 정의하기

### 1) 확률통계: 공분산 행렬
화학 feature 벡터를 확률변수 벡터 \(X \in \mathbb{R}^p\) 라고 보면,

- 평균: \(\mu = \mathbb{E}[X]\)
- 공분산: \(\Sigma = \mathbb{E}[(X-\mu)(X-\mu)^\top]\)

\(\Sigma\)의 **대각 원소**는 각 feature의 분산,  
**비대각 원소** \(\Sigma_{ij}\)는 feature \(i,j\)의 공분산(함께 움직임)입니다.

> “feature 상관이 있다”는 말은 \(\Sigma\)가 대각행렬이 아니라는 뜻(비대각이 0이 아님)으로 엄밀화할 수 있습니다.

### 2) 선형대수: 대칭행렬의 직교대각화(스펙트럴 정리)
공분산 행렬 \(\Sigma\)는 항상 **대칭행렬**입니다.  
스펙트럴 정리에 의해, 어떤 직교행렬 \(V\)와 대각행렬 \(\Lambda\)가 존재해서

\[
\Sigma = V\Lambda V^\top
\]

- \(V\)의 열벡터들이 **고유벡터(eigenvectors)** (서로 직교)
- \(\Lambda\)의 대각이 **고유값(eigenvalues)**

### 3) “좌표계 재정의” = 고유벡터 기저로의 변환
새 좌표를 다음처럼 정의합니다.

\[
Z = V^\top (X-\mu)
\]

그러면

\[
\mathrm{Cov}(Z) = V^\top \Sigma V = \Lambda
\]

즉 \(\mathrm{Cov}(Z)\)가 **대각행렬**이므로, \(Z\)의 각 좌표(PC score)들은 **서로 공분산이 0**입니다.  
이게 “PCA로 상관관계를 없앤다”의 수학적 의미입니다.

- **PC 축**: 고유벡터
- **PC별 분산**: 고유값

> 주의: 공분산 0(비상관) =/= 완전 독립(independent)  
> PCA는 “선형 상관”을 없애는 것이지, 확률적 독립까지 보장하진 않습니다.

### 4) 역행렬/행렬식은 PCA에서 어디에?
- 행렬식(det)은 교과서에서 고유값을 “정의”할 때 등장합니다:
  \(\det(\Sigma-\lambda I)=0\)
- 역행렬(inv)은 **회귀(OLS)**에서 직접 등장합니다:
  \(\hat\beta=(X^\top X)^{-1}X^\top y\)  
  feature들이 강하게 상관되면 \(X^\top X\)가 **거의 특이(ill-conditioned)**해져서 \((\cdot)^{-1}\)이 수치적으로 불안정해집니다(=다중공선성 문제).
- PCA는 축이 직교이므로, 변환 기저의 역행렬은 \(V^{-1}=V^\top\)로 단순해집니다.

---

## 실험 요약(너의 목표 흐름)
1. train(175×231)에서 스케일링(권장) + PCA를 fit  
2. 저장된 축으로 train/test를 동일하게 transform  
3. 변환된 데이터(PC score)로 회귀모델 학습/평가  
4. (선택) **lossless 차원 축소**: train에서 정보 손실 없이 줄일 수 있는 최소 차원 = rank(\(X_c\)) ≤ \(n-1\)

아래부터 코드를 실행하세요.


In [1]:

import os
import json
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
import joblib

import matplotlib.pyplot as plt


In [2]:

# =========================
# 0) 사용자 설정(여기만 수정)
# =========================

# (A) 데이터 로딩 방식 2가지 중 하나를 선택
# 1) separate_files: train.csv / test.csv가 따로 있음
# 2) single_file: 한 파일에 train/test가 같이 있고 split 컬럼으로 구분됨
DATA_MODE = "separate_files"  # "separate_files" or "single_file"

# --- separate_files 모드 ---
TRAIN_PATH = "./train.csv"
TEST_PATH  = "./test.csv"     # test에 y가 없으면(=평가 못함) None으로 두세요: TEST_PATH=None

# --- single_file 모드 ---
FULL_PATH  = "./data.csv"
SPLIT_COL  = "split"          # 예: 'split' 컬럼 값이 'train'/'test'
TRAIN_TAG  = "train"
TEST_TAG   = "test"

# (B) 타깃(관능평가) 컬럼 지정
# - 가장 안전: 직접 리스트로 지정
TARGET_COLS = [
    # 예시: "aroma", "flavor", "overall"
]

# (C) feature 컬럼 지정(선택)
# - None이면: "TARGET_COLS를 제외한 숫자형 컬럼"을 feature로 자동 선택
FEATURE_COLS = None

# (D) 분석에서 제외할 컬럼(아이디/메타데이터 등)
DROP_COLS = []  # 예: ["sample_id", "beer_name"]

# (E) 스케일링 + PCA 설정
STANDARDIZE_BEFORE_PCA = True   # 화학 변수 단위/스케일이 다르면 True 권장
PCA_SVD_SOLVER = "full"         # "full" 권장(결정적, p>n에서도 안정적)
RANK_TOL = 1e-12                # "0이 아닌 고유값" 판정 허용 오차

# PCA에서 몇 차원(k)을 쓸지 선택하는 방식
# - "lossless_rank": train에서 정보손실 없이 줄이는 최소 차원(=rank)
# - "evr_threshold": 누적 설명분산(EVR) 기준
# - "manual": 직접 지정
# - "cv_search": k를 CV로 찾아서 예측 성능 최적화
N_COMPONENTS_MODE = "lossless_rank"  # "lossless_rank" | "evr_threshold" | "manual" | "cv_search"
EVR_THRESHOLD = 0.99
MANUAL_K = 30

# (F) 회귀모델 설정(기본은 Ridge: 다중공선성에 비교적 강함)
MODEL_NAME = "ridge"  # "linreg" | "ridge" | "elasticnet"
RIDGE_ALPHA = 1.0
ELASTICNET_ALPHA = 0.01
ELASTICNET_L1_RATIO = 0.2

# (G) CV 설정(모델/차원 선택)
N_SPLITS = 5
RANDOM_STATE = 42

# (H) 저장 경로
ARTIFACT_DIR = "./artifacts"
PCA_ARTIFACT_PATH = os.path.join(ARTIFACT_DIR, "pca_axes.joblib")
MODEL_ARTIFACT_PATH = os.path.join(ARTIFACT_DIR, "regression_model.joblib")
PREDICTION_OUT_PATH = os.path.join(ARTIFACT_DIR, "test_predictions.csv")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("✅ 설정 로드 완료")


✅ 설정 로드 완료


In [3]:

# =========================
# 1) 데이터 로드
# =========================

def load_data() -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    if DATA_MODE == "separate_files":
        train_df = pd.read_csv(TRAIN_PATH)
        test_df = pd.read_csv(TEST_PATH) if TEST_PATH is not None else None
        return train_df, test_df

    elif DATA_MODE == "single_file":
        df = pd.read_csv(FULL_PATH)
        if SPLIT_COL not in df.columns:
            raise ValueError(f"SPLIT_COL='{SPLIT_COL}' 이(가) 데이터에 없습니다. 현재 컬럼: {list(df.columns)[:30]} ...")
        train_df = df[df[SPLIT_COL] == TRAIN_TAG].copy()
        test_df = df[df[SPLIT_COL] == TEST_TAG].copy()
        return train_df, test_df

    else:
        raise ValueError("DATA_MODE는 'separate_files' 또는 'single_file' 이어야 합니다.")

train_df, test_df = load_data()

print(f"train_df shape: {train_df.shape}")
if test_df is not None:
    print(f"test_df  shape: {test_df.shape}")
else:
    print("test_df: None (test 평가를 건너뜁니다)")


FileNotFoundError: [Errno 2] No such file or directory: './train.csv'

In [ ]:

# =========================
# 2) feature/target 분리
# =========================

def infer_feature_cols(df: pd.DataFrame, target_cols: List[str], drop_cols: List[str]) -> List[str]:
    drop_set = set(drop_cols)
    drop_set |= set(target_cols)
    if DATA_MODE == "single_file":
        drop_set.add(SPLIT_COL)

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    feature_cols = [c for c in numeric_cols if c not in drop_set]
    return feature_cols

# drop 컬럼 제거(있으면)
train_df = train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns])
if test_df is not None:
    test_df = test_df.drop(columns=[c for c in DROP_COLS if c in test_df.columns])

# 타깃 컬럼 유효성 체크(있으면)
missing_targets = [c for c in TARGET_COLS if c not in train_df.columns]
if len(missing_targets) > 0:
    raise ValueError(f"TARGET_COLS 중 train에 없는 컬럼이 있습니다: {missing_targets}")

if FEATURE_COLS is None:
    feature_cols = infer_feature_cols(train_df, TARGET_COLS, DROP_COLS)
else:
    feature_cols = FEATURE_COLS

# feature 컬럼 존재 체크
missing_features = [c for c in feature_cols if c not in train_df.columns]
if len(missing_features) > 0:
    raise ValueError(f"FEATURE_COLS 중 train에 없는 컬럼이 있습니다: {missing_features[:20]} ...")

# X, y 분리
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COLS].copy() if len(TARGET_COLS) > 0 else None

if test_df is not None:
    # test에도 같은 feature가 있어야 함
    missing_features_test = [c for c in feature_cols if c not in test_df.columns]
    if len(missing_features_test) > 0:
        raise ValueError(f"test에 feature 컬럼이 없습니다: {missing_features_test[:20]} ...")
    X_test = test_df[feature_cols].copy()

    # test에 y가 있으면 평가 가능
    y_test = test_df[TARGET_COLS].copy() if (len(TARGET_COLS) > 0 and all(c in test_df.columns for c in TARGET_COLS)) else None
else:
    X_test, y_test = None, None

print(f"#features = {len(feature_cols)}")
print("X_train:", X_train.shape)
print("y_train:", None if y_train is None else y_train.shape)


In [ ]:

# =========================
# 3) 결측치 처리(필요 시)
# =========================
# PCA/회귀는 결측치를 그대로 두면 에러가 나는 경우가 많습니다.
# 여기서는 가장 기본적인 "train 평균으로 대체"를 사용합니다.
# 더 좋은 방식(KNN impute 등)이 필요하면 이 블록을 바꾸세요.

def mean_impute_train_test(X_train: pd.DataFrame, X_test: Optional[pd.DataFrame]=None) -> Tuple[pd.DataFrame, Optional[pd.DataFrame], pd.Series]:
    col_means = X_train.mean(axis=0)
    X_train_imp = X_train.fillna(col_means)
    X_test_imp = None
    if X_test is not None:
        X_test_imp = X_test.fillna(col_means)
    return X_train_imp, X_test_imp, col_means

missing_train = X_train.isna().sum().sum()
missing_test = X_test.isna().sum().sum() if X_test is not None else 0
print(f"missing values - train: {missing_train}, test: {missing_test}")

X_train, X_test, impute_means = mean_impute_train_test(X_train, X_test)


In [ ]:

# =========================
# 4) PCA fit (train만!)
# =========================

# (1) 스케일러: 평균 제거는 PCA에 사실상 필수, 표준화는 옵션
scaler = StandardScaler(with_mean=True, with_std=STANDARDIZE_BEFORE_PCA)
X_train_scaled = scaler.fit_transform(X_train)

# (2) p>n인 경우에도 full SVD가 안정적
# sklearn은 n_components=None이면 min(n_samples, n_features)까지 반환합니다.
pca_full = PCA(n_components=None, svd_solver=PCA_SVD_SOLVER)
pca_full.fit(X_train_scaled)

explained = pca_full.explained_variance_
explained_ratio = pca_full.explained_variance_ratio_
cum_ratio = np.cumsum(explained_ratio)

# "lossless rank" (train 기준): 0이 아닌 고유값(=분산)이 있는 PC 개수
k_lossless = int(np.sum(explained > RANK_TOL))
k_evr = int(np.searchsorted(cum_ratio, EVR_THRESHOLD) + 1)

print(f"pca_full components returned: {pca_full.n_components_}")
print(f"k_lossless (rank-based, tol={RANK_TOL}): {k_lossless}")
print(f"k_evr (cum EVR>={EVR_THRESHOLD}): {k_evr}")

# 실제로 선택할 k
if N_COMPONENTS_MODE == "lossless_rank":
    k = k_lossless
elif N_COMPONENTS_MODE == "evr_threshold":
    k = k_evr
elif N_COMPONENTS_MODE == "manual":
    k = MANUAL_K
elif N_COMPONENTS_MODE == "cv_search":
    k = None  # 아래에서 CV로 선택
else:
    raise ValueError("N_COMPONENTS_MODE가 올바르지 않습니다.")

print("✅ PCA 준비 완료")


In [ ]:

# =========================
# 5) 설명분산(EVR) 시각화
# =========================
plt.figure(figsize=(7,4))
plt.plot(explained_ratio, marker="o")
plt.title("Explained Variance Ratio per PC (train)")
plt.xlabel("PC index (0-based)")
plt.ylabel("Explained variance ratio")
plt.grid(True)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(cum_ratio, marker="o")
plt.axhline(EVR_THRESHOLD, linestyle="--")
plt.title("Cumulative Explained Variance Ratio (train)")
plt.xlabel("Number of PCs (0-based index on x)")
plt.ylabel("Cumulative explained variance ratio")
plt.grid(True)
plt.show()


In [ ]:

# =========================
# 6) 선택한 k로 PCA 변환 + "비상관" 확인
# =========================

def fit_transform_pca(X_train_scaled: np.ndarray, X_test_scaled: Optional[np.ndarray], n_components: int) -> Tuple[PCA, np.ndarray, Optional[np.ndarray]]:
    pca = PCA(n_components=n_components, svd_solver=PCA_SVD_SOLVER)
    Z_train = pca.fit_transform(X_train_scaled)
    Z_test = pca.transform(X_test_scaled) if X_test_scaled is not None else None
    return pca, Z_train, Z_test

def max_abs_offdiag_cov(Z: np.ndarray) -> float:
    C = np.cov(Z, rowvar=False, ddof=1)
    off = C - np.diag(np.diag(C))
    return float(np.max(np.abs(off)))

# test도 스케일링(단, train에서 fit된 scaler로!)
X_test_scaled = scaler.transform(X_test) if X_test is not None else None

if k is not None:
    pca, Z_train, Z_test = fit_transform_pca(X_train_scaled, X_test_scaled, k)
    print("Z_train:", Z_train.shape)
    if Z_test is not None:
        print("Z_test:", Z_test.shape)

    print("max |off-diagonal covariance| in Z_train:",
          max_abs_offdiag_cov(Z_train))
else:
    pca = None
    Z_train, Z_test = None, None
    print("N_COMPONENTS_MODE='cv_search' 이므로 여기서는 변환을 건너뜁니다.")


In [ ]:

# =========================
# 7) (선택) 재구성 오차 확인: train에서 100% 표현이 되나?
# =========================
# - PCA(k_lossless) + inverse_transform을 하면,
#   (수치 오차 제외) train의 스케일된 X를 거의 완벽하게 복원할 수 있어야 합니다.

def reconstruction_rmse(X_true: np.ndarray, X_hat: np.ndarray) -> float:
    return float(np.sqrt(np.mean((X_true - X_hat) ** 2)))

if k is not None:
    X_train_scaled_hat = pca.inverse_transform(Z_train)
    rmse_scaled = reconstruction_rmse(X_train_scaled, X_train_scaled_hat)

    # 원래 스케일로도 복원
    X_train_hat = scaler.inverse_transform(X_train_scaled_hat)
    rmse_raw = reconstruction_rmse(X_train.values, X_train_hat)

    print(f"Reconstruction RMSE (scaled space): {rmse_scaled:.6e}")
    print(f"Reconstruction RMSE (raw space):    {rmse_raw:.6e}")

    # k가 lossless인지 감각적으로 확인
    print(f"현재 k={k}, k_lossless={k_lossless}")
else:
    print("cv_search 모드에서는 k가 아직 정해지지 않았습니다.")


In [ ]:

# =========================
# 8) PCA 축/스케일러 저장 (중요)
# =========================
# - train에서 학습된 scaler(평균/표준편차)와 PCA 축(components)을 저장해두면
#   이후 새로운 데이터도 동일한 축으로 변환할 수 있습니다.

if k is not None:
    artifact = {
        "feature_cols": feature_cols,
        "target_cols": TARGET_COLS,
        "drop_cols": DROP_COLS,
        "standardize_before_pca": STANDARDIZE_BEFORE_PCA,
        "impute_means": impute_means,   # 결측 대체용
        "scaler": scaler,
        "pca": pca,
        "k": k,
        "explained_variance_ratio": explained_ratio[:pca.n_components_],
        "explained_variance": explained[:pca.n_components_],
    }
    joblib.dump(artifact, PCA_ARTIFACT_PATH)
    print("✅ PCA artifact saved to:", PCA_ARTIFACT_PATH)
else:
    print("cv_search 모드: k 선택 후에 저장하세요.")


In [ ]:

# =========================
# 9) 회귀 모델 학습/평가 (Baseline vs PCA(PCR))
# =========================

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred, multioutput="uniform_average")))

def r2(y_true, y_pred) -> float:
    return float(r2_score(y_true, y_pred, multioutput="uniform_average"))

def per_target_report(y_true_df: pd.DataFrame, y_pred: np.ndarray, prefix: str="") -> pd.DataFrame:
    rows = []
    for i, col in enumerate(y_true_df.columns):
        yt = y_true_df[col].values
        yp = y_pred[:, i]
        rows.append({
            "target": col,
            f"{prefix}RMSE": np.sqrt(mean_squared_error(yt, yp)),
            f"{prefix}R2": r2_score(yt, yp),
        })
    return pd.DataFrame(rows)

def make_model(name: str):
    name = name.lower()
    if name == "linreg":
        return LinearRegression()
    if name == "ridge":
        return Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE)
    if name == "elasticnet":
        return ElasticNet(alpha=ELASTICNET_ALPHA, l1_ratio=ELASTICNET_L1_RATIO, random_state=RANDOM_STATE)
    raise ValueError("MODEL_NAME must be one of: linreg, ridge, elasticnet")

if y_train is None:
    raise ValueError("TARGET_COLS가 비어있어서 회귀를 할 수 없습니다. 관능평가 컬럼명을 TARGET_COLS에 넣어주세요.")

# ---------- Baseline: (스케일링만) 원본 feature로 회귀 ----------
baseline_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("model", make_model(MODEL_NAME)),
])

baseline_pipe.fit(X_train, y_train)
baseline_pred_train = baseline_pipe.predict(X_train)

print("Baseline (raw features) - Train")
print("  RMSE:", rmse(y_train, baseline_pred_train))
print("  R2:  ", r2(y_train, baseline_pred_train))

# test가 있으면 평가
if X_test is not None and y_test is not None:
    baseline_pred_test = baseline_pipe.predict(X_test)
    print("Baseline (raw features) - Test")
    print("  RMSE:", rmse(y_test, baseline_pred_test))
    print("  R2:  ", r2(y_test, baseline_pred_test))

# ---------- PCR: PCA score로 회귀 ----------
if k is not None:
    pcr_pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=STANDARDIZE_BEFORE_PCA)),
        ("pca", PCA(n_components=k, svd_solver=PCA_SVD_SOLVER)),
        ("model", make_model(MODEL_NAME)),
    ])

    pcr_pipe.fit(X_train, y_train)
    pcr_pred_train = pcr_pipe.predict(X_train)

    print(f"\nPCR (PCA k={k}) - Train")
    print("  RMSE:", rmse(y_train, pcr_pred_train))
    print("  R2:  ", r2(y_train, pcr_pred_train))

    if X_test is not None and y_test is not None:
        pcr_pred_test = pcr_pipe.predict(X_test)
        print(f"PCR (PCA k={k}) - Test")
        print("  RMSE:", rmse(y_test, pcr_pred_test))
        print("  R2:  ", r2(y_test, pcr_pred_test))

else:
    print("cv_search 모드: 아래 셀에서 k를 CV로 고른 뒤 PCR을 학습하세요.")


In [ ]:

# =========================
# 10) (선택) k를 CV로 선택 (예측성능 최적화)
# =========================
# PCA의 목적이 "비상관 축"일 뿐 아니라 "예측"까지라면,
# 설명분산만 보고 k를 정하기보다 CV로 k를 고르는 것이 안전합니다.

if N_COMPONENTS_MODE != "cv_search":
    print("N_COMPONENTS_MODE가 'cv_search'가 아니므로 이 셀은 선택 실행입니다.")
else:
    max_k = min(X_train.shape[0] - 1, X_train.shape[1])
    # 적당한 k 후보들(원하면 바꾸세요)
    k_grid = sorted(set([2, 5, 10, 20, 30, 40, 60, 80, 120, max_k]))
    k_grid = [kk for kk in k_grid if kk <= max_k and kk >= 1]

    param_grid = {"pca__n_components": k_grid}

    if MODEL_NAME.lower() == "ridge":
        alpha_grid = np.logspace(-3, 3, 13)
        param_grid["model__alpha"] = alpha_grid
    elif MODEL_NAME.lower() == "elasticnet":
        alpha_grid = np.logspace(-4, 1, 10)
        l1_grid = [0.1, 0.2, 0.5, 0.8, 0.95]
        param_grid["model__alpha"] = alpha_grid
        param_grid["model__l1_ratio"] = l1_grid

    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scorer = make_scorer(rmse, greater_is_better=False)  # 음수 RMSE를 최대화 == RMSE 최소화

    search_pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=STANDARDIZE_BEFORE_PCA)),
        ("pca", PCA(svd_solver=PCA_SVD_SOLVER)),
        ("model", make_model(MODEL_NAME)),
    ])

    gs = GridSearchCV(
        estimator=search_pipe,
        param_grid=param_grid,
        scoring=scorer,
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )
    gs.fit(X_train, y_train)

    print("Best params:", gs.best_params_)
    print("Best CV score (neg RMSE):", gs.best_score_)

    best_pipe = gs.best_estimator_
    joblib.dump(best_pipe, MODEL_ARTIFACT_PATH)
    print("✅ Best pipeline saved to:", MODEL_ARTIFACT_PATH)

    # Train / Test 평가
    pred_train = best_pipe.predict(X_train)
    print("\nBest CV-chosen PCR - Train")
    print("  RMSE:", rmse(y_train, pred_train))
    print("  R2:  ", r2(y_train, pred_train))

    if X_test is not None:
        pred_test = best_pipe.predict(X_test)
        if y_test is not None:
            print("Best CV-chosen PCR - Test")
            print("  RMSE:", rmse(y_test, pred_test))
            print("  R2:  ", r2(y_test, pred_test))
        else:
            # test에 y가 없으면 예측만 저장
            out = pd.DataFrame(pred_test, columns=TARGET_COLS)
            out.to_csv(PREDICTION_OUT_PATH, index=False)
            print("✅ test predictions saved to:", PREDICTION_OUT_PATH)


In [ ]:

# =========================
# 11) 저장한 PCA 축을 다시 불러와서 transform 하는 예시
# =========================

loaded = joblib.load(PCA_ARTIFACT_PATH)
loaded_feature_cols = loaded["feature_cols"]
loaded_scaler = loaded["scaler"]
loaded_pca = loaded["pca"]
loaded_means = loaded["impute_means"]

def max_abs_offdiag_cov(Z: np.ndarray) -> float:
    C = np.cov(Z, rowvar=False, ddof=1)
    off = C - np.diag(np.diag(C))
    return float(np.max(np.abs(off)))

def pca_transform(df: pd.DataFrame) -> np.ndarray:
    # 1) feature 컬럼만 추출 (순서 중요!)
    X = df[loaded_feature_cols].copy()
    # 2) 결측치 처리(저장된 train 평균으로)
    X = X.fillna(loaded_means)
    # 3) 스케일링(train 기준)
    Xs = loaded_scaler.transform(X)
    # 4) PCA projection
    Z = loaded_pca.transform(Xs)
    return Z

# 예: train_df를 다시 변환해보기
Z_again = pca_transform(train_df)
print("Z_again shape:", Z_again.shape)
print("max |off-diagonal covariance|:", max_abs_offdiag_cov(Z_again))
